# 03 · Build the HBIM `Account` subject area (aligned to FIBO)

HBIM = the bank's **Harmonised Business Information Model**. We turn the mapped
terms into an OWL ontology whose classes are linked to FIBO via
`owl:equivalentClass` / `rdfs:subClassOf` / `skos:closeMatch`. Preferred-term
attributes become `skos:prefLabel` / `skos:altLabel` / `skos:definition`, mapped
relations become OWL restrictions reusing FIBO properties, and one sample
individual is added so notebook 04 can infer instance-level facts.

Reads the extraction + mapping, writes `output/hbim_account.ttl`.

In [1]:
import sys, json
from pathlib import Path

# Make ../src importable so we can reuse the shared paths + namespaces.
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
from common import (COLLIBRA_EXPORT, EXTRACTED_FILE, MAPPING_FILE,
                    HBIM_TTL, HBIM_INFERRED_TTL, FIBO_EXCERPT, OUTPUT_DIR,
                    PREFIXES, bind_all,
                    CAA, FPAS, FSE, REL, CMNS_ID, CMNS_ORG, HBIM)
import pandas as pd
pd.set_option("display.max_colwidth", 60)
print("Project root:", SRC.parent)

Project root: C:\Users\marci\OneDrive\DEV\EDU\AIML\Graph ML\Ontology Engineering\Ontology Repository\FIBO\Ontology Mapping


In [2]:
from rdflib import Graph, Literal, RDF, RDFS, OWL, URIRef, BNode
from rdflib.namespace import SKOS

with open(EXTRACTED_FILE, encoding="utf-8") as fh:
    extracted = json.load(fh)
with open(MAPPING_FILE, encoding="utf-8") as fh:
    mapping = json.load(fh)

pref = extracted["preferred_attributes"]
by_id = {m["collibra_id"]: m for m in mapping["term_mappings"]}

## Helpers

In [3]:
def slug(name):
    "'Current Account' -> 'CurrentAccount'"
    return "".join(w.capitalize() for w in name.replace("-", " ").split())

def restriction(g, cls, prop, filler):
    r = BNode()
    g.add((r, RDF.type, OWL.Restriction))
    g.add((r, OWL.onProperty, prop))
    g.add((r, OWL.someValuesFrom, filler))
    g.add((cls, RDFS.subClassOf, r))

## Build the graph

In [4]:
g = bind_all(Graph()); g.bind("skos", SKOS)
onto = URIRef("https://example.org/hbim/account")
g.add((onto, RDF.type, OWL.Ontology))
g.add((onto, RDFS.label, Literal("HBIM - Account subject area (aligned to FIBO)")))

hbim_class_for = {}
for term in extracted["business_terms"]:
    tid = term["id"]; p = pref[tid]
    if not p["is_preferred"]:          # non-preferred synonyms folded in below
        continue
    cls = HBIM[slug(p["preferred_label"] or term["name"])]
    hbim_class_for[tid] = cls
    g.add((cls, RDF.type, OWL.Class))
    g.add((cls, SKOS.prefLabel, Literal(p["preferred_label"] or term["name"], lang="en")))
    if p["definition"]:
        g.add((cls, SKOS.definition, Literal(p["definition"], lang="en")))
    if p["acronym"]:
        g.add((cls, SKOS.altLabel, Literal(p["acronym"], lang="en")))
    for syn in p["synonyms"]:
        g.add((cls, SKOS.altLabel, Literal(syn, lang="en")))

    m = by_id.get(tid)
    if m:
        fibo = URIRef(m["fibo_iri"]); kind = m["mapping_kind"]
        if kind == "exact":
            g.add((cls, OWL.equivalentClass, fibo))
        elif kind == "narrower":
            g.add((cls, RDFS.subClassOf, fibo))
        else:
            g.add((cls, SKOS.closeMatch, fibo))

# fold non-preferred synonyms onto their preferred class as altLabels
for r in extracted["relations"]:
    if r["type"] == "is synonym of":
        pc = hbim_class_for.get(r["source_id"])
        other = next((a for a in extracted["business_terms"] if a["id"] == r["target_id"]), None)
        if pc is not None and other is not None:
            g.add((pc, SKOS.altLabel, Literal(other["name"], lang="en")))

# relation axioms (reuse FIBO object properties)
acct = hbim_class_for.get("bt-account")
if acct and "bt-account-holder" in hbim_class_for:
    restriction(g, acct, REL.isHeldBy, hbim_class_for["bt-account-holder"])
if acct and "bt-account-identifier" in hbim_class_for:
    restriction(g, acct, CMNS_ID.isIdentifiedBy, hbim_class_for["bt-account-identifier"])

# sample individual -> instance-level inferences in notebook 04
inst = HBIM["account-GB29NWBK60161331926819"]
g.add((inst, RDF.type, hbim_class_for["bt-current-account"]))
g.add((inst, RDFS.label, Literal("Customer current account GB29 NWBK ...6819")))

print(f"{len(set(g.subjects(RDF.type, OWL.Class)))} classes, {len(g)} triples")

7 classes, 51 triples


## Serialize and preview the HBIM ontology

In [5]:
g.serialize(destination=HBIM_TTL, format="turtle")
print("[written]", HBIM_TTL, "\n")
print(g.serialize(format="turtle"))

[written] C:\Users\marci\OneDrive\DEV\EDU\AIML\Graph ML\Ontology Engineering\Ontology Repository\FIBO\Ontology Mapping\output\hbim_account.ttl 

@prefix caa: <https://spec.edmcouncil.org/fibo/ontology/FBC/ProductsAndServices/ClientsAndAccounts/> .
@prefix cmns-id: <https://www.omg.org/spec/Commons/Identifiers/> .
@prefix hbim: <https://example.org/hbim/account/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix rel: <https://spec.edmcouncil.org/fibo/ontology/FND/Relations/Relations/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .

<https://example.org/hbim/account> a owl:Ontology ;
    rdfs:label "HBIM - Account subject area (aligned to FIBO)" .

hbim:Account a owl:Class ;
    rdfs:subClassOf [ a owl:Restriction ;
            owl:onProperty rel:isHeldBy ;
            owl:someValuesFrom hbim:AccountHolder ],
        [ a owl:Restriction ;
            owl:onProperty cmns-id:isIdentifiedBy ;
            owl:someValue